In [ ]:
// Интерфейсы для имитации множественного наследования
public interface ICancelable
{
    void CancelItem(LineItem lineItem, string cancellationReason);
}

public interface ITaxable
{
    decimal CalculateTax();
}

public class LineItem
{
    public string Description { get; set; }
    public decimal Quantity { get; set; }
    public decimal UnitPrice { get; set; }
    public decimal Total => Quantity * UnitPrice;
    public DateTime? SupplyDate { get; set; }
    public string CancellationReason { get; set; } = string.Empty;
    public bool IsReturned { get; set; }
    
    // Дополнительное поле для налога
    public decimal Tax { get; set; }

    public LineItem(string description, decimal quantity, decimal unitPrice)
    {
        Description = description;
        Quantity = quantity;
        UnitPrice = unitPrice;
    }

    public decimal CalculateTax(decimal taxRate)
    {
        return Total * taxRate;
    }
}

public class Invoice
{
    public string InvoiceNumber { get; protected set; }
    public DateTime IssueDate { get; protected set; }
    public decimal TotalAmount { get; protected set; }
    public string Currency { get; set; } = "USD"; // Новое поле для валюты
    public decimal Discount { get; set; } = 0; // Новое поле для скидки

    protected List<LineItem> LineItems { get; } = new List<LineItem>();

    public Invoice(string invoiceNumber, DateTime issueDate)
    {
        InvoiceNumber = invoiceNumber;
        IssueDate = issueDate;
    }

    public virtual void CalculateTotal()
    {
        TotalAmount = LineItems.Sum(item => item.Total);
        TotalAmount -= Discount;  // Применение скидки
    }

    public virtual void AddLine(LineItem lineItem)
    {
        LineItems.Add(lineItem);
        CalculateTotal();
    }

    public virtual void RemoveLine(LineItem lineItem)
    {
        LineItems.Remove(lineItem);
        CalculateTotal();
    }

    // Метод для применения скидки
    public void ApplyDiscount(decimal discountAmount)
    {
        Discount = discountAmount;
        CalculateTotal();
    }

    // Метод для изменения валюты
    public void ChangeCurrency(string newCurrency)
    {
        Currency = newCurrency;
        Console.WriteLine($"Currency changed to {Currency}");
    }

    public ReadOnlyCollection<LineItem> GetLineItems()
    {
        return new ReadOnlyCollection<LineItem>(LineItems);
    }

    public void PrintInfo()
    {
        Console.WriteLine($"Invoice #{InvoiceNumber} | Date: {IssueDate:dd.MM.yyyy} | Total: {TotalAmount:C} | Currency: {Currency}");
    }
}

// Класс GoodsInvoice с добавлением новых методов и атрибутов
public class GoodsInvoice : Invoice, ITaxable
{
    public DateTime SupplyDate { get; set; }

    public GoodsInvoice(string invoiceNumber, DateTime issueDate, DateTime supplyDate)
        : base(invoiceNumber, issueDate)
    {
        SupplyDate = supplyDate;
    }

    public override void AddLine(LineItem lineItem)
    {
        lineItem.SupplyDate = SupplyDate;
        base.AddLine(lineItem);
        Console.WriteLine($"[GoodsInvoice] Item added: {lineItem.Description}. Supply date set: {SupplyDate:dd.MM.yyyy}");
    }

    public decimal CalculateTax()
    {
        // Применяем 15% налог для товаров
        return LineItems.Sum(item => item.CalculateTax(0.15m));
    }
}

// Класс ServiceInvoice с добавлением новых методов и атрибутов
public class ServiceInvoice : Invoice, ICancelable
{
    public DateTime ServiceDate { get; set; }

    private readonly Dictionary<LineItem, string> _cancellationLog = new Dictionary<LineItem, string>();

    public string PendingCancellationReason { get; set; } = string.Empty;

    public ServiceInvoice(string invoiceNumber, DateTime issueDate, DateTime serviceDate)
        : base(invoiceNumber, issueDate)
    {
        ServiceDate = serviceDate;
    }

    public override void RemoveLine(LineItem lineItem)
    {
        if (string.IsNullOrWhiteSpace(PendingCancellationReason))
            PendingCancellationReason = "Reason not specified";

        _cancellationLog[lineItem] = PendingCancellationReason;
        lineItem.CancellationReason = PendingCancellationReason;

        base.RemoveLine(lineItem);
        Console.WriteLine($"[ServiceInvoice] Service removed: {lineItem.Description}. Reason: {PendingCancellationReason}");
        PendingCancellationReason = string.Empty;
    }

    public void CancelItem(LineItem lineItem, string cancellationReason)
    {
        lineItem.CancellationReason = cancellationReason;
        _cancellationLog[lineItem] = cancellationReason;
        Console.WriteLine($"[ServiceInvoice] Item '{lineItem.Description}' cancelled. Reason: {cancellationReason}");
    }

    public void PrintCancellationLog()
    {
        Console.WriteLine("Cancellation Log:");
        foreach (var entry in _cancellationLog)
            Console.WriteLine($"  - {entry.Key.Description}: {entry.Value}");
    }
}

// Класс CombinedInvoice с учетом возвратов и дополнительным методом
public class CombinedInvoice : Invoice
{
    public bool ReturnAllowed { get; set; }

    public CombinedInvoice(string invoiceNumber, DateTime issueDate, bool returnAllowed)
        : base(invoiceNumber, issueDate)
    {
        ReturnAllowed = returnAllowed;
    }

    public override void CalculateTotal()
    {
        if (!ReturnAllowed)
        {
            base.CalculateTotal();
            Console.WriteLine("[CombinedInvoice] Returns disabled. Standard calculation applied.");
            return;
        }

        decimal fullAmount = LineItems.Sum(item => item.Total);
        decimal returnedAmount = LineItems.Where(item => item.IsReturned).Sum(item => item.Total);
        TotalAmount = fullAmount - returnedAmount;

        Console.WriteLine($"[CombinedInvoice] Returns allowed. Adjusted total: {TotalAmount:C} (Returned: {returnedAmount:C})");
    }
}

// Новый класс ExtendedInvoice для демонстрации сложного наследования
public class ExtendedInvoice : GoodsInvoice, ICancelable
{
    public ExtendedInvoice(string invoiceNumber, DateTime issueDate, DateTime supplyDate)
        : base(invoiceNumber, issueDate, supplyDate)
    { }

    public void CancelItem(LineItem lineItem, string cancellationReason)
    {
        Console.WriteLine($"[ExtendedInvoice] Item '{lineItem.Description}' cancelled. Reason: {cancellationReason}");
    }
}

public class InvoiceProcessor
{
    public void ProcessBatch(IEnumerable<Invoice> invoices)
    {
        foreach (var invoice in invoices)
        {
            invoice.CalculateTotal();
            invoice.PrintInfo();
        }
    }

    public bool TransferLineItem(Invoice source, Invoice destination, LineItem item)
    {
        if (!source.GetLineItems().Contains(item))
        {
            Console.WriteLine($"[Processor] Item '{item.Description}' not found in source invoice {source.InvoiceNumber}.");
            return false;
        }

        source.RemoveLine(item);
        destination.AddLine(item);
        Console.WriteLine($"[Processor] Transferred '{item.Description}' from {source.InvoiceNumber} to {destination.InvoiceNumber}");
        return true;
    }

    public decimal CalculateGrandTotal(IEnumerable<Invoice> invoices)
    {
        return invoices.Sum(inv => inv.TotalAmount);
    }
}

class Program
{
    static void Main()
    {
        // Пример работы с новыми функциями
        var goodsInv = new GoodsInvoice("G-001", new DateTime(2026, 4, 20), new DateTime(2026, 4, 25));
        var serviceInv = new ServiceInvoice("S-002", new DateTime(2026, 4, 20), new DateTime(2026, 4, 22));
        var combinedInv = new CombinedInvoice("C-003", new DateTime(2026, 4, 20), true);

        // Создаем позиции
        var laptop = new LineItem("Laptop Pro", 2, 1500m);
        var consulting = new LineItem("IT Consulting", 10, 200m);
        var cables = new LineItem("Network Cables", 50, 10m);
        var monitor = new LineItem("Monitor 27", 1, 400m) { IsReturned = true };

        // Добавляем позиции
        goodsInv.AddLine(laptop);
        serviceInv.AddLine(consulting);
        combinedInv.AddLine(cables);
        combinedInv.AddLine(monitor);

        // Печатаем информацию о счетах
        var processor = new InvoiceProcessor();
        List<Invoice> batch = new List<Invoice> { goodsInv, serviceInv, combinedInv };
        processor.ProcessBatch(batch);

        // Применение скидки и изменение валюты
        goodsInv.ApplyDiscount(200);
        goodsInv.ChangeCurrency("EUR");

        // Финальные итоги
        goodsInv.PrintInfo();
        serviceInv.PrintInfo();
        combinedInv.PrintInfo();
        
        decimal grandTotal = processor.CalculateGrandTotal(batch);
        Console.WriteLine($"\nGrand Total (all invoices): {grandTotal:C}");
    }
}